# Detailed Checklist — Sportsbook Spread Prediction & Similar Projects

Markdown checklists — click into a cell and toggle `[ ]` to `[x]`, or just track progress visually. Organized in four parts:

1. **This project's step-by-step checklist**
2. **Transferable checklist** for any similar market-comparison regression project
3. **Common pitfalls checklist** (including betting-market-specific leakage traps)
4. **Definition-of-done checklist** for the whole project


## Part 1 — This Project's Step-by-Step Checklist

### Step 1 — Load & Inspect
- [ ] Import pandas, numpy, matplotlib, seaborn
- [ ] Load `sports_betting_odds.csv` into `df`
- [ ] Print `df.shape`, `df.head()`, `df.info()`
- [ ] Run `df.describe(include='all').T`
- [ ] Identify every numeric-looking text column (units, `%`, `$`, `"W-L"` records)


### Step 2 — Data Quality Audit
- [ ] Build a dtype/nunique/missing audit table
- [ ] Check `df.duplicated().sum()`; inspect flagged rows before dropping
- [ ] Scan every text column's `.unique()` for disguised-missing sentinels (`"Not Reported"`, `"Unknown"`)
- [ ] Normalize `sportsbook` casing before encoding


### Step 3 — Records & Scoring Stats
- [ ] Parse `home_record`/`away_record` (`"W-L"`) into win percentages
- [ ] Create `home_games_played`/`away_games_played`; drop the original record columns
- [ ] Strip `" pts"` from `home_ppg`, `away_ppg`, `home_papg`, `away_papg`; cast to float


### Step 4 — Rest, Injuries, Travel & Market Context
- [ ] Strip `" days"` from rest-day columns; cast to int
- [ ] Clean injury columns: strip `" players"`, handle `"Not Reported"` sentinel, **add a `*_unreported` flag**, impute
- [ ] Clean travel distance: strip comma + `" miles"`, handle `"Unknown"` sentinel, **add a `travel_unknown` flag**, impute
- [ ] Strip `"%"` from all percentage columns; cast to int
- [ ] Strip `"$"`/`"M"` from `betting_volume`; cast to float
- [ ] Convert `"Yes"/"No"` flag columns to `0/1`


### Step 5 — Domain-Derived Differential Features
- [ ] `net_rating_home`, `net_rating_away`, `net_rating_diff`
- [ ] `win_pct_diff`
- [ ] `rest_advantage`
- [ ] `injury_advantage`
- [ ] `star_out_diff`
- [ ] `sharp_public_divergence`
- [ ] One-sentence justification written for each new feature


### Step 6 — Advanced EDA & the Leakage Audit (the step unique to this project type)
- [ ] Target distribution plotted; skewness checked
- [ ] Correlation heatmap built **including** `opening_spread_home` and `closing_moneyline_home`
- [ ] Explicit correlation values computed between the target and each market-quote column
- [ ] Written paragraph explaining why each flagged column is/isn't safe to use as a feature, given this project's specific goal
- [ ] Decision made and documented: which columns are excluded from the feature set
- [ ] VIF computed on the retained feature set
- [ ] Boxplots of target by `home_back_to_back` and `home_star_out`
- [ ] IQR outlier check on the target; explicit keep/remove decision


### Step 7 — Leakage-Safe Encoding
- [ ] Categorical columns bucketed by cardinality (`sportsbook` low, `home_team`/`away_team` high)
- [ ] Train/test split performed **before** any target-encoding statistic is computed
- [ ] Target-encoding maps fit on train only; applied to test with an unseen-category fallback
- [ ] Zero `NaN`s confirmed in `X_train`/`X_test`
- [ ] Market-quote and outcome columns confirmed excluded from `X`


### Step 8 — Baseline & Linear Models
- [ ] Mean-predictor baseline scored
- [ ] `LinearRegression` scored
- [ ] `Ridge` scored; coefficients compared


### Step 9 — Tree-Ensemble Models
- [ ] `RandomForestRegressor` (default) scored
- [ ] `GradientBoostingRegressor` (default) scored
- [ ] All models collected into one comparison table, sorted by MAE
- [ ] Result checked against expectation: is this an additive or interaction-heavy problem, and does the winning model match that?


### Step 10 — Cross-Validation & Tuning
- [ ] 5-fold CV MAE computed for the leading candidate (mean ± std reported)
- [ ] Hyperparameter distribution defined for the chosen model family
- [ ] `RandomizedSearchCV` run with `cv=5`
- [ ] Best params and best CV score reported
- [ ] Tuned model refit and evaluated once on the held-out test set


### Step 11 — Evaluation & Diagnostics
- [ ] MAE, RMSE, R² reported together
- [ ] MAPE reported with an explicit caveat about instability near a 0-point spread (or omitted)
- [ ] Predicted-vs-actual scatterplot
- [ ] Residuals-vs-predicted scatterplot checked for heteroscedasticity


### Step 12 — Feature Importance
- [ ] Impurity-based or coefficient-based top-10 plotted
- [ ] Permutation importance top-10 plotted
- [ ] Agreement between the two checked against basketball/domain intuition


### Step 13 — Betting Backtest
- [ ] `edge = model_prediction - opening_spread_home` computed on the test set
- [ ] Decision rule (threshold + bet side) defined **before** looking at results
- [ ] ATS outcome computed from `home_margin` and `closing_spread_home` (never used as training features)
- [ ] Win rate and simulated ROI (at standard -110 odds) computed
- [ ] Compared against "bet every game" baseline and the -110 breakeven win rate (52.4%)
- [ ] Written caveats included: sample size, synthetic data, no bet-sizing/market-impact modeling


### Step 14 — Persistence & Inference
- [ ] Final model saved with `joblib.dump`
- [ ] Encoding maps + column order saved alongside the model
- [ ] `predict_fair_spread()` function written, reproducing every training-time transform
- [ ] Function tested on 2–3 made-up matchups with plausible outputs


### Step 15 — Conclusions
- [ ] Final model choice stated with a one-line justification
- [ ] Expected error margin stated in points
- [ ] Top 3–5 spread drivers named
- [ ] Backtest result stated with at least one reason for caution
- [ ] At least one limitation and one next step named


## Part 2 — Transferable Checklist for Market-Comparison Regression Projects

### Phase 1 — Understand the Problem
- [ ] Target variable and the "market quote" you're being compared against are both clearly identified
- [ ] Project goal stated precisely: "predict the market's number as accurately as possible" vs. "build an independent estimate to find disagreement" (these require different feature-inclusion decisions)
- [ ] Cost asymmetry / decision consequences considered (false positive vs. false negative cost)


### Phase 2 — Data Acquisition & Audit
- [ ] Shape, dtypes, nulls, duplicates all checked
- [ ] Every combined-format text column identified (e.g. `"W-L"` records) and a parsing plan written
- [ ] Sentinel values and disguised missing data identified, with a flag-column plan
- [ ] Category-casing inconsistencies checked


### Phase 3 — Feature Engineering
- [ ] Unit-bearing strings converted to numerics, idempotently
- [ ] **Differential features constructed between the two compared entities**
- [ ] Missing-value strategy chosen per column, with a `*_missing`/`*_unreported` flag preserved where the missingness itself might be informative
- [ ] Every transformation documented for reproducibility


### Phase 4 — The Market-Quote Leakage Audit (do this explicitly, every time)
- [ ] Every market-quote-style column identified and listed
- [ ] Correlation of each with the target checked; anything unexpectedly high (>0.9) investigated
- [ ] For each market-quote column: written decision on inclusion, tied to the project's stated goal (Phase 1)
- [ ] VIF checked on the retained feature set


### Phase 5 — Encoding & Splitting
- [ ] Categorical columns partitioned by cardinality
- [ ] Split performed **before** any target-dependent computation
- [ ] Target encoding fit on train only; unseen test categories handled with a fallback
- [ ] `random_state` fixed everywhere


### Phase 6 — Modeling
- [ ] Trivial baseline established
- [ ] At least one linear model AND at least one tree ensemble fit and fairly compared
- [ ] No assumption made in advance about which model family will win
- [ ] Train vs. test metrics compared to detect over/underfitting


### Phase 7 — Evaluation, Interpretation & Backtesting
- [ ] Metrics appropriate to the target's scale and distribution chosen (flag MAPE instability if the target can be near zero)
- [ ] Feature importance computed via two independent methods
- [ ] If a downstream decision exists (bet/trade/price adjustment): decision rule pre-registered, backtested against genuinely held-out outcomes, compared to an explicit baseline
- [ ] Backtest results interpreted with appropriate statistical caution (sample size, overfitting risk, synthetic-vs-real data)


### Phase 8 — Communicate & Persist
- [ ] Findings summarized in plain language, with a stated confidence level
- [ ] Model + preprocessing artifacts persisted together
- [ ] Limitations and next steps explicitly named


## Part 3 — Common Pitfalls Checklist

- [ ] **Market-quote leakage** — a different-format snapshot of the same market assessment (e.g. moneyline vs. spread) included as a feature
- [ ] **Goal-relative leakage** — a technically-legitimate, pre-event column (e.g. opening line) included as a feature despite defeating the project's stated "independent estimate" goal
- [ ] **Outcome leakage** — a post-event column (final score, game result) accidentally left in the feature set instead of reserved for backtesting only
- [ ] **Backtest overfitting** — a decision-rule threshold chosen *after* seeing which one produces the best-looking backtest result
- [ ] **Small-sample overconfidence** — treating a good-looking backtest on a few dozen/hundred bets as proof of a real edge
- [ ] **MAPE misuse** — reporting MAPE on a target that can be near zero without flagging its instability
- [ ] **Assuming ensembles always win** — skipping the linear-model comparison because "tree ensembles are usually better"
- [ ] **Discarding missingness signal** — imputing a disguised-missing sentinel without also keeping a flag column
- [ ] **Category-casing fragmentation** — leaving inconsistent text casing to silently multiply one-hot columns
- [ ] **Model without its preprocessing** — persisting a model without the encoding maps/column order needed to use it on new data
- [ ] **Presenting synthetic-data results as real-world validated** — forgetting to caveat that a demonstration dataset's specific numbers don't transfer to real markets


## Part 4 — Definition-of-Done Checklist for the Whole Project

- [ ] Every raw column is either numeric, properly encoded, or intentionally dropped with a stated reason
- [ ] No `NaN` values remain anywhere in the final training/test feature matrices
- [ ] Every market-quote-style column has an explicit, written inclusion/exclusion decision
- [ ] At least one linear and one tree-ensemble model trained and fairly compared
- [ ] Final model selected with a written justification, not just "best number"
- [ ] Cross-validated performance estimate reported alongside single-split test metrics
- [ ] Two independent feature-importance methods agree on (most of) the top drivers
- [ ] If a decision rule was backtested: the rule was pre-registered, evaluated against genuine held-out outcomes, and compared to a meaningful baseline
- [ ] Model and preprocessing artifacts are persisted and reloadable
- [ ] An inference function exists that takes raw, uncleaned input and returns a prediction
- [ ] A plain-language summary exists, including explicit caveats about data limitations
